# 00-baseline — 압축 없이 통과시키기

압축을 하지 않는 랩입니다. 두 가지를 위해 있습니다.

1. **기준선** — 다른 랩의 "30% 절감" 이 무엇 대비인지 정해 줍니다
2. **하네스 검증** — 압축을 안 했으니 절감은 0%, 보존율은 100% 여야 합니다.
   아니면 압축기가 아니라 **측정 도구가 고장 난 것**입니다

먼저 원리를 한 단계씩 보고, 마지막에 `configs/` 의 모든 조건을 돌립니다.

## 1. kit 불러오기

랩은 `labs/` 를 경로에 넣고 `from kit import ...` 로 씁니다.

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

# 노트북을 켜 둔 채로 저장소를 갱신하면 커널이 **예전 코드를 물고 있습니다.**
# 그러면 새로 생긴 함수가 없다는 에러(AttributeError)가 나는데, 원인이 코드가
# 아니라 커널이라 찾기가 어렵습니다. 그래서 이 셀을 돌릴 때마다 새로 읽습니다.
_stale = [m for m in list(sys.modules)
          if m == "kit" or m.startswith("kit.")
          or m in ("transforms", "blocks", "summarize", "compress")]
for _m in _stale:
    del sys.modules[_m]

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)
if _stale:
    print(f"모듈 {len(_stale)}개를 새로 읽었습니다 — 커널에 남아 있던 예전 코드를 지웠습니다")

print("배포명 기본값:", env.get("AZURE_OPENAI_DEPLOYMENT", "(없음)"))
print("엔드포인트    :", env.mask_endpoint(env.get("AZURE_OPENAI_ENDPOINT")))

## 2. 설정 읽기

설정은 코드가 아니라 yaml 에 둡니다. 그래야 `runs/` 경로에 조건 이름이
그대로 남아서, 나중에 무엇을 돌렸는지 알 수 있습니다.

In [ ]:
cfg = C.load("configs/noop.yaml")

table(
    ["항목", "값"],
    [["name", cfg.name], ["lab", cfg.lab],
     ["params", cfg.params or "(없음)"],
     ["dataset.path", Path(cfg.dataset["path"]).name],
     ["model", cfg.model],
     ["tokenizer", cfg.tokenizer or "(기본: local)"]],
    align=["left", "left"], title="설정",
)

## 3. 토큰을 어떻게 셀 것인가

**세 가지 중에 고르실 수 있고, 기본값은 `both` 입니다.**

| mode | 기준값 | 호출 | 언제 쓰나요 |
|---|---|---|---|
| **`both`** (기본) | API 실측 | 텍스트당 1회 | 과금 기준으로 재면서 추정 오차도 같이 봅니다 |
| `api` | API 실측 | 텍스트당 1회 | 실측만 필요할 때 |
| `local` | tiktoken | 없음 | 네트워크·자격증명 없이 돌릴 때 |

`both` 가 기본인 이유는, **실제로 돈이 나가는 기준은 API 응답의
`usage.input_tokens`** 이기 때문입니다. 그렇다고 tiktoken 을 버리면 추정이
얼마나 어긋나는지 알 수 없어서, 둘을 같이 재고 차이를 남깁니다.

```yaml
tokenizer:
  mode: both           # both | api | local
  deployment:          # 비우면 .env 의 AZURE_OPENAI_DEPLOYMENT
  cache: true          # 같은 텍스트는 한 번만 호출
```

명령줄에서 덮어쓰실 수도 있습니다.

```bash
python compress.py configs/noop.yaml --tokenizer local
```

**자격증명이 없으면** `both` 는 로컬 계산으로 내려가되 왜 그랬는지 알려줍니다.
`api` 는 같은 상황에서 예외를 냅니다 — 실측이 꼭 필요하다고 선언한 것이라
조용히 다른 값을 드리면 안 되기 때문입니다.

**`api` 를 쓰실 때 캐시를 끄지 말아주세요.** 케이스 N건이면 압축 전후로
2N 회를 부르고, 스윕을 10단계 돌리면 그만큼 곱해집니다. 결과는
`kit/.cache/` 에 남아 다음 실행부터 재사용됩니다.

> **주의** — `api` 값에는 메시지 포맷 오버헤드가 포함됩니다(실측 +6).
> 압축 전후를 같은 방식으로 재므로 **비율 비교는 안전하지만**,
> `local` 값과 나란히 놓으면 안 됩니다. 그래서 `token_backend` 를 기록합니다.

In [ ]:
# 여기서 방식을 바꿔 보실 수 있습니다. None 이면 config 값을 씁니다.
MODE = None          # None | "both" | "api" | "local"

spec = dict(cfg.tokenizer)
if MODE:
    spec["mode"] = MODE

counter = T.make_counter(spec, cfg.model)
print("측정 방식:", counter.backend)
print("설명    :", counter.describe())

sample = "환불 수수료는 결제금액의 10%입니다."
print(f"\n예시 {len(sample)}자 → {counter(sample):,} 토큰")

지금 설정(`noop.yaml`)은 `local` 이라 위 셀에서는 **API 를 부르지 않았습니다.**
말로만 읽으면 두 방식의 차이가 잘 안 와닿으니, 실제로 한 번 불러서
같은 텍스트를 두 방식으로 재보겠습니다.

**호출은 텍스트당 1회씩, 아래 3건이 전부입니다.** `.env` 가 없으면 건너뜁니다.

In [ ]:
probe = ["환불 수수료는 결제금액의 10%입니다.",
         "주문번호 A-1003 은 환불 완료 상태입니다.",
         ("제7조 환불 시 결제금액의 10%를 수수료로 공제한다. "
          "제9조 단, 결제 후 7일 이내 취소는 수수료를 면제한다. "
          "제10조 분쟁은 서울중앙지방법원을 관할로 한다.")]

local_c = T.make_counter({"mode": "local"}, cfg.model)

try:
    api_c = T.make_counter(
        {"mode": "api", "deployment": env.get("AZURE_OPENAI_DEPLOYMENT"),
         "cache": True}, cfg.model)
    rows = []
    for t in probe:
        a, b = local_c(t), api_c(t)
        rows.append([t[:30] + ("…" if len(t) > 30 else ""),
                     f"{len(t)}자", f"{a:,}", f"{b:,}", f"{b - a:+d}"])
    api_c.save()

    table(
        ["텍스트", "길이", "local", "api", "차이"],
        rows,
        align=["left", "right", "right", "right", "right"],
        title="같은 텍스트를 두 방식으로",
        note="차이가 어느 텍스트에서나 같은 값이면 그건 내용이 아니라 "
             "메시지 포맷 오버헤드입니다.",
    )
    print(api_c.describe())
    print("\n차이가 일정한 이유 — 요청을 보낼 때 역할 구분자 같은 것이 붙습니다.")
    print("텍스트가 길든 짧든 똑같이 붙으므로 상수만큼 차이가 납니다.")
except Exception as e:
    print(f"api 측정을 건너뜁니다 — {type(e).__name__}: {str(e)[:160]}")
    print("\n자격증명이 있으면 아래로 준비하실 수 있습니다.")
    print("  cd labs && cp .env.example .env")
    print("없어도 이 노트북의 나머지는 local 로 전부 돌아갑니다.")

## 4. 코퍼스 살펴보기

`must_include` 는 **정답에 꼭 필요한 문자열**입니다. 이게 있어야 보존율을
잴 수 있고, LLM 호출이 없으므로 스윕을 수백 번 돌려도 비용이 0 입니다.

In [ ]:
cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
info = dataset.summarize(cases)

table(
    ["항목", "값"],
    [["케이스", f'{info["n_cases"]}건'],
     ["총 길이", f'{info["n_chars"]:,}자'],
     ["must_include 보유", f'{info["with_must_include"]}건'],
     ["유형", ", ".join(f"{k}×{v}" for k, v in info["kinds"].items())]],
    align=["left", "left"], title="코퍼스",
)

c = cases[0]
print(f"\n[{c.id}] {c.kind}")
print(f"질문        : {c.question}")
print(f"must_include: {c.must_include}")
print(f"원문        : {c.text[:80]}…")

## 5. 압축 — 이 랩은 그대로 통과시킵니다

**모든 랩이 이 시그니처를 씁니다.** 랩을 바꾼다는 건 이 함수 하나를
바꾼다는 뜻이고, 나머지(코퍼스·지표·기록)는 전부 재사용됩니다.

In [ ]:
def compress(text: str, **params) -> tuple[str, dict]:
    """압축하지 않습니다. 반환은 (압축문, 메타) 입니다."""
    return text, {}


after, meta = compress(cases[0].text)
print("원문과 동일:", after == cases[0].text)

## 6. 집계 — 평균만 보면 안 됩니다

**정답 보존율**은 각 케이스에서 `must_include` 문자열 중 압축 후에도 남은
비율입니다. 100% 면 하나도 안 잃은 것입니다.

집계는 평균과 함께 **최저값**과 **유형별 분해**를 항상 냅니다.

> 평균 90% 여도 한 케이스가 0% 면 그 질문에는 아예 답할 수 없습니다.
> 평균은 그걸 가립니다. **최저 보존율부터 보세요.**

In [ ]:
records = [
    metrics.per_case(c.id, c.kind, c.text, compress(c.text, **cfg.params)[0],
                     c.must_include, counter)
    for c in cases
]
m = metrics.aggregate(records, counter)

table(
    ["지표", "값", "정상값", "뜻"],
    [["절감률", pct(m["saved"]), "0.0%", "압축을 안 했으므로"],
     ["토큰", f'{m["tokens_before"]:,} → {m["tokens_after"]:,}', "변화 없음", ""],
     ["평균 보존율", pct(m.get("survival_mean")), "100%", "전체 케이스 평균"],
     ["하위 5%", pct(m.get("survival_p5")), "100%", "나쁜 쪽 5% 지점"],
     ["최저 보존율", pct(m.get("survival_worst")), "100%", "가장 많이 깨진 케이스"],
     ["측정 방식", m["token_backend"], "local 또는 api", "다르면 비교 불가"]],
    align=["left", "right", "right", "left"], title="집계",
)

## 7. 하네스 자가 점검

압축을 안 했으니 아래가 성립해야 합니다. 어긋나면 압축기가 아니라
**측정 도구가 고장 난 것**이고, 그 상태로는 다른 랩의 숫자를 믿을 수 없습니다.

In [ ]:
problems = []
if m["saved"] != 0.0:
    problems.append(f'절감률이 0 이 아닙니다 ({m["saved"]:.2%}) — 로더가 원문을 바꾸고 있습니다')
if m.get("survival_worst") not in (None, 1.0):
    problems.append(f'최저 보존율이 100% 가 아닙니다 ({m["survival_worst"]:.1%})')

if problems:
    print("하네스 점검 실패")
    for p in problems:
        print("  ✗", p)
else:
    print("하네스 정상 — 다른 랩을 돌려도 됩니다.")

## 8. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/00-baseline/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

이 랩에는 조건이 둘 있습니다.

| 설정 | 토큰 측정 | 비용 |
|---|---|---|
| `noop.yaml` | `local` (tiktoken) | 0 |
| `noop-api.yaml` | `api` (모델 호출 실측) | 텍스트당 1회, 캐시되면 0 |

`api` 조건은 `.env` 가 없으면 건너뜁니다. 건너뛴 이유를 표에 남겨서,
"돌았는데 결과가 없는" 상태와 "아예 못 돌린" 상태를 구분합니다.

> **"호출 0회" 가 나와도 놀라지 마세요.** 같은 텍스트를 이미 잰 적이 있으면
> 디스크 캐시에서 꺼내 씁니다. 처음 한 번은 실제로 12회를 부르고 20초 남짓
> 걸립니다. 강제로 다시 부르시려면 설정에 `refresh: true` 를 넣거나
> 명령줄에서 `--refresh-tokens` 를 쓰시면 됩니다.

In [ ]:
def run_config(path):
    """설정 하나를 끝까지 돌리고 (설정, 지표, 결과경로) 를 돌려줍니다."""
    cfg = C.load(path)
    cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    counter = T.make_counter(cfg.tokenizer, cfg.model)

    run = Run(cfg, RUNS)
    for c in cases:
        after, extra = compress(c.text, **cfg.params)
        run.add(metrics.per_case(c.id, c.kind, c.text, after,
                                 c.must_include, counter, extra),
                before=c.text, after=after)

    m = metrics.aggregate(run.records, counter)
    m["dataset_name"] = Path(cfg.dataset["path"]).name
    counter.save()
    if counter.stats():
        m["token_calls"] = counter.stats()
    out = run.finish(m, ["압축 없음. 다른 랩의 절감률은 이 결과를 기준으로 읽습니다.",
                         counter.describe()])
    return cfg, m, out, counter


results, skipped = [], []
for p in sorted(Path("configs").glob("*.yaml")):
    try:
        cfg, m, out, cnt = run_config(p)
        results.append((cfg.name, m, out))
        print(f'{p.name:20s} 절감 {m["saved"]:6.1%} · {cnt.describe()}')
    except Exception as e:
        skipped.append((p.name, f"{type(e).__name__}: {e}"))
        print(f"{p.name:20s} 건너뜀 — {type(e).__name__}: {str(e)[:80]}")

if skipped:
    print("\n건너뛴 설정이 있습니다. 자격증명이 없으면 api 조건은 못 돌립니다.")
    print("  cd labs && cp .env.example .env   (또는 cp ../scripts/explore/.env .env)")

## 9. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

`local` 과 `api` 의 토큰 수가 다른 것이 정상입니다. 차이는 **메시지 포맷
오버헤드**(케이스당 +6)이고, 이건 텍스트가 아니라 역할 구분자 같은
프레이밍입니다. 그래서 두 값을 나란히 놓고 "어느 쪽이 맞다" 를 따지면 안 됩니다.

In [ ]:
table(
    ["설정", "측정 방식", "건수", "토큰", "절감", "최저 보존율"],
    [[n, m["token_backend"], m["n"],
      f'{m["tokens_before"]:,} → {m["tokens_after"]:,}',
      pct(m["saved"]), pct(m.get("survival_worst"))]
     for n, m, _ in results],
    align=["left", "left", "right", "right", "right", "right"],
    title="조건 비교",
    note="절감 0% · 최저 보존율 100% 가 두 조건 모두에서 나와야 하네스가 정상입니다.",
)

if len(results) == 2:
    a, b = (m for _, m, _ in results)
    diff = abs(a["tokens_before"] - b["tokens_before"])
    print(f"측정 방식 차이: {diff:,} 토큰 ({diff / max(a["n"], 1):.1f}/건)")
    print("케이스당 +6 이면 메시지 포맷 오버헤드입니다 — 텍스트가 아니라 프레이밍입니다.")

## 10. 진짜로 줄었나 — API 응답으로 확인하기

여기까지의 절감률은 전부 **tiktoken 추정치**입니다. 실제로 청구되는 값은
API 응답의 `usage.input_tokens` 이고, 둘이 항상 같지는 않습니다.

| 왜 어긋나나 | 얼마나 |
|---|---|
| 메시지 포맷 오버헤드 (역할 구분자 등) | 텍스트당 상수 (실측 +6) |
| 배포 모델의 토크나이저가 tiktoken 과 다를 수 있음 | 모델마다 |
| 압축 결과의 특수 문자를 모델이 어떻게 쪼개는지 | **해봐야 압니다** |

마지막 줄이 중요합니다. 이 랩은 압축을 안 하므로 **양쪽 다 0% 가 나와야 정상**입니다. 여기서 0% 가 아니면 측정 경로 어딘가가 원문을 바꾸고 있다는 뜻입니다.

그래서 몇 건만 뽑아 **압축 전과 후를 각각 실제로 보내보고**, 응답이 알려주는
토큰 수로 절감률을 다시 계산합니다. 호출은 케이스당 2회이고 캐시됩니다.

In [ ]:
from kit import verify

DEPLOY = env.get("AZURE_OPENAI_DEPLOYMENT")

try:
    pairs = [(c.id, c.text, compress(c.text, **cfg.params)[0])
             for c in cases]
    r = verify.billed(pairs, deployment=DEPLOY, model=DEPLOY, limit=3)
    t = r["totals"]

    table(
        ["케이스", "tiktoken 전→후", "API 실측 전→후", "추정 절감", "실측 절감"],
        [[x["id"],
          f'{x["local_before"]:,} → {x["local_after"]:,}',
          f'{x["api_before"]:,} → {x["api_after"]:,}',
          pct(x["local_saved"]), pct(x["api_saved"])]
         for x in r["rows"]],
        foot=["합계",
              f'{t["local_before"]:,} → {t["local_after"]:,}',
              f'{t["api_before"]:,} → {t["api_after"]:,}',
              pct(t["local_saved"]), pct(t["api_saved"])],
        align=["left", "right", "right", "right", "right"],
        title=f'과금 기준으로 다시 재기 ({t["n"]}건)',
        note="'API 실측' 은 응답의 usage.input_tokens 를 그대로 읽은 값입니다.",
    )

    print(verify.verdict(t))
    print(f'텍스트당 오버헤드 {t["overhead_per_text"]:+.1f} 토큰 — '
          f'역할 구분자 같은 프레이밍이라 길이와 무관하게 붙습니다.')
    print(r["counter"].describe())
except Exception as e:
    print(f"과금 검증을 건너뜁니다 — {type(e).__name__}: {str(e)[:160]}")
    print("\\n자격증명이 있으면 아래로 준비하실 수 있습니다.")
    print("  cd labs && cp .env.example .env")
    print("없어도 위까지의 결과는 전부 유효합니다. 다만 추정치입니다.")

## 정리

- **기준선이 없으면 절감률은 의미가 없습니다** — 무엇 대비인지가 있어야 합니다
- **하네스를 먼저 검증합니다** — `kit` 을 고친 뒤에는 항상 여기부터 돌리세요
- **정답 보존율은 평균 대신 최저값** — 평균은 한 케이스의 붕괴를 가립니다
- **측정 방식을 기록합니다** — `local` 과 `api` 는 값이 다르므로 섞으면 안 됩니다

### 다음 랩

[`01-lossless-structure`](../01-lossless-structure/run.ipynb) — 의미를 하나도
안 버리고 표현만 바꿉니다. `compress()` 만 바뀌고 나머지는 그대로입니다.